## PPO LOSS

In [ ]:
import torch
def compute_loss(self,model,inputs,return_outputs=False,num_item_in_batch=None):
    prompt_ids,prompt_mask=inputs['input_ids'],inputs['prompt_mask']
    completion_ids,completion_mask=inputs['completion_ids'],inputs["completion_mask"]
    input_ids=torch.cat([prompt_ids,completion_ids],dim=1)
    attention_mask= torch.cat([prompt_mask,completion_mask])
    logits_to_keep =completion_ids.size(1) #只需要计算completion 的token的loss
    per_token_logps= self._get_per_token_logps(model,input_ids,attention_mask, logits_to_keep)
    ref_per_token_logps=inputs["ref_per_token_logps"]
    #Loss = E[min(ratio * advantage, clip(ratio, 1-ε, 1+ε) * advantage)]
    per_token_kl= torch.exp(ref_per_token_logps-per_token_logps)-(ref_per_token_logps-per_token_logps)-1
    advantages=inputs["advantages"]
    # x-x.detach
    # log_ratio = per_token_logps - per_token_logps.detach()  # log(π_new / π_old)
    # ratio = torch.exp(log_ratio)  # π_new / π_old
    per_token_loss=torch.exp(per_token_logps-per_token_logps.detach())* advantages.unsqueeze(-1)
    per_token_loss=-(per_token_loss-self.beta*per_token_kl)
    loss=((per_token_loss * completion_mask).sum(dim=1)/completion_mask.sum(dim=1)).mean()


## GRPO LOSS

In [ ]:
import einops
from einops import rearrange
def compute_loss(self,model,inputs,mask,rewards_func):
    completion_mask=torch.cat([inputs["prompt_mask"],inputs["completion_mask"]],dim=1)
    rewards= rewards_func.sum(dim=1)# b* g 标量
    per_token_logps=self._get_per_token_logps(model,inputs,mask,logit_to_keep=True)
    ref_per_token_logps=inputs["ref_per_token_logps"]

    mean_group_rewards=rearrange(rewards,"(b g) -> b g",g=self.num_generation)\
        .mean(dim=1) \
        .repeat_interleave(self.num_generation,dim=0)
    std_group_rewards=rearrange(rewards,"(b g) -> b g",g=self.num_generation)\
        .std(dim=1) \
        .repeat_interleave(self.num_generation,dim=0)
    advantages=(rewards-mean_group_rewards)/(std_group_rewards+1e-4)   
    per_token_kl=torch.exp(ref_per_token_logps-per_token_logps)-(ref_per_token_logps-per_token_logps)-1
    important_ratio=torch.exp(per_token_logps-per_token_logps.detach())
    per_token_loss=torch.min(important_ratio*advantages.unsqueeze(1),torch.clamp(important_ratio,1-self.epsilon,1+self.epsilon)*advantages.unsqueeze(1))
    per_token_loss=-(per_token_loss-self.beta*per_token_kl)
    loss=(per_token_loss*completion_mask).sum(dim=1)/completion_mask.sum(dim=1).mean()
    return loss

In [ ]:
import einops
from einops import rearrange
def compute_loss(self,model,inputs,mask,rewards_func):
    completion_mask=torch.cat([inputs["prompt_mask"],inputs["completion_mask"]],dim=1)
    rewards= rewards_func.sum(dim=1)# b* g 标量
    per_token_logps=self._get_per_token_logps(model,inputs,mask,logit_to_keep=True)
    ref_per_token_logps=inputs["ref_per_token_logps"]

    mean_group_rewards=rearrange(rewards,"(b g) -> b g",g=self.num_generation)\
        .mean(dim=1) \
        .repeat_interleave(self.num_generation,dim=0)
    std_group_rewards=rearrange(rewards,"(b g) -> b g",g=self.num_generation)\
        .std(dim=1) \
        .repeat_interleave(self.num_generation,dim=0)
    advantages=(rewards-mean_group_rewards)/(std_group_rewards+1e-4)   
    per_token_kl=torch.exp(ref_per_token_logps-per_token_logps)-(ref_per_token_logps-per_token_logps)-1
    pi_pi_old=torch.exp(per_token_logps-per_token_logps.detach())
    per_token_loss=torch.min(pi_pi_old*advantages.unsqueeze(1),torch.clamp(pi_pi_old,1-self.epsilon,1+self.epsilon)*advantages.unsqueeze(1))
    per_token_loss=-(per_token_loss-self.beta*per_token_kl)
    loss=(per_token_loss*completion_mask).sum(dim=1)/completion_mask.sum(dim=1).mean()
    return loss

In [ ]:
beta=0.1
def compute_loss(self,model,inputs,mask,rewards_func):
    mask=torch.cat(inputs["prompt_mask"],inputs["completion_mask"],dim=1)
    rewards=rewards_func.sum(dim=1)
    per_token_logps=self.model()
    per_token_ref_logps=self.ref_model()

    # 先拆分，在平均再重复
    mean_group_rewards=rearrange(rewards,"(b g) -> b g",g=self.num_generation)\
        .mean(dim=1)\
        .repeat_interleave(self.num_generation,dim=0)
    std_group_rewards=rearrange(rewards,"(b g) -> b g",g=self.num_generation)\
        .mean(dim=1)\
        .repeat_interleave(self.num_generation,dim=0)
    adv=(rewards-mean_group_rewards)/(std_group_rewards+0.000001)
    ratio=torch.exp(per_token_logps-per_token_logps.detach())
    per_token_loss=torch.min(ratio*adv,torch.clamp(ratio,1-self.epsilon,1+self.epsilon)*adv.unsqueeze(1))
    kl=torch.exp(per_token_logps-per_token_ref_logps)-(per_token_logps-per_token_ref_logps)-1
    per_token_loss_kl=-(per_token_loss-kl*beta)
    per_token_loss_mask=(per_token_loss_kl*mask)/mask.sum(dim=-1).mean()
    return per_token_loss_mask
